In [1]:
import numpy as np
import pandas as pd
from modelens import RegressionAnalyzer

In [2]:
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", None)
pd.set_option("display.max_columns", None)

In [3]:
raw_data = pd.read_csv('dataset.csv')
df = raw_data.copy()

#### 1 - EDA + ETL

In [4]:
analyzer = RegressionAnalyzer(df,target='mpg')

In [5]:
analyzer.info()

In [6]:
# Check Invalid Data
df['horsepower'].unique()

<StringArray>
['130', '165', '150', '140', '198', '220', '215', '225', '190', '170', '160',
  '95',  '97',  '85',  '88',  '46',  '87',  '90', '113', '200', '210', '193',
   '?', '100', '105', '175', '153', '180', '110',  '72',  '86',  '70',  '76',
  '65',  '69',  '60',  '80',  '54', '208', '155', '112',  '92', '145', '137',
 '158', '167',  '94', '107', '230',  '49',  '75',  '91', '122',  '67',  '83',
  '78',  '52',  '61',  '93', '148', '129',  '96',  '71',  '98', '115',  '53',
  '81',  '79', '120', '152', '102', '108',  '68',  '58', '149',  '89',  '63',
  '48',  '66', '139', '103', '125', '133', '138', '135', '142',  '77',  '62',
 '132',  '84',  '64',  '74', '116',  '82']
Length: 94, dtype: str

In [7]:
# ? is Invalida Data in horsepower Column
df['horsepower'] = df['horsepower'].replace({"?":np.nan}).astype(float)
df = df.dropna(subset=['horsepower'])
df['horsepower'].unique()

array([130., 165., 150., 140., 198., 220., 215., 225., 190., 170., 160.,
        95.,  97.,  85.,  88.,  46.,  87.,  90., 113., 200., 210., 193.,
       100., 105., 175., 153., 180., 110.,  72.,  86.,  70.,  76.,  65.,
        69.,  60.,  80.,  54., 208., 155., 112.,  92., 145., 137., 158.,
       167.,  94., 107., 230.,  49.,  75.,  91., 122.,  67.,  83.,  78.,
        52.,  61.,  93., 148., 129.,  96.,  71.,  98., 115.,  53.,  81.,
        79., 120., 152., 102., 108.,  68.,  58., 149.,  89.,  63.,  48.,
        66., 139., 103., 125., 133., 138., 135., 142.,  77.,  62., 132.,
        84.,  64.,  74., 116.,  82.])

In [8]:
# One Hot Encoding
df = pd.get_dummies(
    df,
    columns=["origin"],
    prefix="origin",
    drop_first=True,
    dtype=int,
)

In [9]:
# Remove UnUsed Columns
if 'car name' in df.columns:
    df = df.drop(columns=['car name'])

In [10]:
analyzer.reinit(df=df,target='mpg')

In [11]:
analyzer.info()

In [12]:
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin_2,origin_3
0,18.0,8,307.0,130.0,3504,12.0,70,0,0
1,15.0,8,350.0,165.0,3693,11.5,70,0,0
2,18.0,8,318.0,150.0,3436,11.0,70,0,0
3,16.0,8,304.0,150.0,3433,12.0,70,0,0
4,17.0,8,302.0,140.0,3449,10.5,70,0,0


#### 2 - Comparing Models

In [13]:
target = 'mpg'
X = df.drop(columns=[target])
y = df[target]
features = df.columns.drop(target)
features

Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
       'model year', 'origin_2', 'origin_3'],
      dtype='str')

In [14]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from sklearn.ensemble import (
    AdaBoostRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
    RandomForestRegressor,
)
from sklearn.linear_model import (
    ElasticNet,
    Lasso,
    LinearRegression,
    Ridge,
)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

models = {
    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge()),
    ]),

    "Lasso": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Lasso()),
    ]),

    "ElasticNet": Pipeline([
        ("scaler", StandardScaler()),
        ("model", ElasticNet()),
    ]),

    # Distance / Kernel
    "KNN": Pipeline([
        ("scaler",StandardScaler()),
        ("KNN",KNeighborsRegressor())
    ]),
    "SVR":Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVR()),
    ]),

    # Tree
    "Decision Tree": DecisionTreeRegressor(
        random_state=42,
    ),

    # Bagging
    "Random Forest": RandomForestRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "Extra Trees": ExtraTreesRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    # Boosting
    "AdaBoost": AdaBoostRegressor(
        random_state=42,
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42,
    ),

    "Hist Gradient Boosting": HistGradientBoostingRegressor(
        random_state=42,
    ),

    "XGBoost": XGBRegressor(
        random_state=42,
        n_jobs=-1,
    ),

    "LightGBM": LGBMRegressor(
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
    ),

    "CatBoost": CatBoostRegressor(
        random_state=42,
        verbose=0,
    ),
}
compare = analyzer.compare_models(models=models,features=features,export_html=True)

#### 3 - Evaluate Data

In [15]:
# _,suspicious_features=analyzer.correlation(features=features)

In [16]:
# analyzer.vif()

In [17]:
# analyzer.evaluate_single_feature_removal(features=features)

In [18]:
# selected_model = ExtraTreesRegressor()
# analyzer.evaluate_single_feature_removal(model=selected_model,features=features,export_html=True)

In [19]:
# candidates = suspicious_features
# analyzer.evaluate_feature_removal_combinations(features=features,candidates=candidates,export_html=True)

#### سبک ترین مدل با کاهش دقت 2 درصدی

In [20]:
# analyzer.compare_models(models={"model":LinearRegression()},features=['weight', 'model year'])
analyzer.select_model_candidates(results=compare)

,Selection,Model,Test R2,Performance Loss,Performance Loss %,Fit Time (s),Speedup vs Best
0,Max Performance,CatBoost,0.8869 ± 0.0225,0.0000,0.000000,0.9891 ± 0.0704,1.000000
1,Balanced,KNN,0.8543 ± 0.0141,0.0326,3.675724,0.0027 ± 0.0001,366.333333
2,Minimal,KNN,0.8543 ± 0.0141,0.0326,3.675724,0.0027 ± 0.0001,366.333333


In [21]:
# analyzer.permutation_importance(model=selected_model,export_html=True)

In [22]:
# analyzer.check_overfitting(model=selected_model)

In [23]:
# analyzer.residual_analysis(model=selected_model,export_html=True)

In [24]:
param_grid = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": [1.0, "sqrt", 0.7],
}
# analyzer.tune_model(model=selected_model,param_grid=param_grid,export_html=True)

In [25]:
# analyzer.learning_curve(model=selected_model)